In [0]:
%pip install molview

In [0]:
%restart_python

In [0]:
import numpy as np
import pandas as pd
import biotite
import molview as mv
import os

In [0]:
path_pdb_complex = "/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_snake_venom/inputs/gopher_alpha_snake_clean.pdb"
pdb_name = os.path.basename(path_pdb_complex)
pdb_data = open(path_pdb_complex).read()

v = mv.view(width = 840, height = 500)
v.addModel(pdb_data, name = pdb_name)
v.setColorMode("rainbow")
v.setBackgroundColor("#F2F2F2")

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics")
from StrucTools import *
import biotite.structure as struc

atom_array_complex_clean = clean_structure(struc_file_path= path_pdb_complex)
atom_array_complex_clean[2173]

In [0]:
from MotifSampler import MotifSampler

mergable_tolerance = 4
target_chain_id = 'C'
path_input_structure = path_pdb_complex
binder_chain_dict = {
    "A" : {'undesirable_paratope_indices' : [], 'num_motifs' : 8},
    "B" : {'undesirable_paratope_indices' : [19, 60, 61, 62, 99, 101, 106, 108, 110, 145], 'num_motifs' : 4},
    "D" : {'undesirable_paratope_indices' : [], 'num_motifs' : 4}
}
motif_sampler = MotifSampler(path_input_structure = path_input_structure, target_chain_id = target_chain_id, binder_chain_dict = binder_chain_dict,mergable_tolerance = mergable_tolerance)
rfd_contig = motif_sampler.sample_motifs_from_all_binder_chains()
rfd_contig

In [0]:
# D = GOPHER motif chain, A = subunit A of snake, B = subunit B of snake, C = target (alpha2-beta1 integrin), D = GOPHER motif chain, E = Ligand
binder_chain_id = 'A' 
target_chain_id = 'C'
desired_epitope_residues = []
cutoff = 6
dict_contact_a_c = determine_binding_interface(pdb_file_path= path_pdb_complex, desired_epitope_residues= desired_epitope_residues, 
                            binder_chain_id= binder_chain_id, target_chain_id= target_chain_id, cutoff= cutoff)
dict_contact_a_c

In [0]:
# D = GOPHER motif chain, A = subunit A of snake, B = subunit B of snake, C = target (alpha2-beta1 integrin), D = GOPHER motif chain, E = Ligand
binder_chain_id = 'D' 
target_chain_id = 'C'
desired_epitope_residues = []
cutoff = 6
dict_contact_d_c = determine_binding_interface(pdb_file_path= path_pdb_complex, desired_epitope_residues= desired_epitope_residues, 
                            binder_chain_id= binder_chain_id, target_chain_id= target_chain_id, cutoff= cutoff)
dict_contact_d_c

In [0]:
hotspot_str = dict_contact_a_c['epitope_indices'] + ',' + dict_contact_d_c['epitope_indices']
hotspot_list = hotspot_str.split(',')
hotspot_list = [int(x) for x in hotspot_list]
unique_hotspot_list = sorted(list(set(hotspot_list)))
hotspot_str = ','.join([f"{target_chain_id}{x}" for x in unique_hotspot_list])
hotspot_str

In [0]:
from CreateRFDYaml import CreateRFDYaml

rfd_yaml = CreateRFDYaml()
rfd_yaml.update_template_dict(pipeline_step = 'rfdiffusion', hyperparameter = 'pdb', value = path_pdb_complex)
rfd_yaml.update_template_dict(pipeline_step = 'rfdiffusion', hyperparameter = 'contigs', value = rfd_contig)
rfd_yaml.update_template_dict(pipeline_step = 'rfdiffusion', hyperparameter = 'hotspot', value = hotspot_str)